In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fix random seed for total reproducibility
SEED = 42

# Load datasets
train_df = pd.read_csv('/kaggle/input/competitions/the-portal-fare-prophecy/train.csv')  # Adjust path if needed
test_df = pd.read_csv('/kaggle/input/competitions/the-portal-fare-prophecy/test.csv')

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {test_df.shape}")

# Preview data types and missing values
print("\nTrain Data Info:")
train_df.info()
display(train_df.head())
# Check your column names
print("Train columns:", train_df.columns.tolist())

Train Shape: (8000, 17)
Test Shape: (2000, 17)

Train Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorID               8000 non-null   int64  
 1   tpep_pickup_datetime   8000 non-null   object 
 2   tpep_dropoff_datetime  8000 non-null   object 
 3   passenger_count        7710 non-null   float64
 4   trip_distance          8000 non-null   float64
 5   RatecodeID             7710 non-null   float64
 6   store_and_fwd_flag     7710 non-null   object 
 7   PULocationID           8000 non-null   int64  
 8   DOLocationID           8000 non-null   int64  
 9   payment_type           8000 non-null   object 
 10  extra                  8000 non-null   float64
 11  tip_amount             8000 non-null   float64
 12  tolls_amount           8000 non-null   float64
 13  improvement_surcharge  8000 non-null   floa

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,extra,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,1,2023-06-29 19:30:24,2023-06-29 18:24:35,1.0,1.45,1.0,N,58,82,Credit Card,2.5,8.105869,0.0,1.0,25.68,2.5,0.0
1,1,2023-06-28 16:52:43,2023-06-28 17:20:05,6.0,0.98,1.0,N,253,153,Cash,2.5,0.570916,0.0,1.0,15.10,2.5,0.0
2,0,2023-06-30 06:03:32,2023-06-30 05:53:56,1.0,0.40,1.0,N,85,264,Credit Card,2.5,4.616720,0.0,1.0,10.40,2.5,0.0
3,1,2023-06-30 14:09:29,2023-06-30 13:07:32,1.0,0.85,1.0,N,2,139,Credit Card,0.0,3.380739,0.0,1.0,12.32,2.5,0.0
4,1,2023-06-30 00:35:04,2023-06-29 23:44:36,NaN,4.59,NaN,NaN,12,99,Wallet,0.0,1.534365,0.0,1.0,27.56,NaN,NaN


Train columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'extra', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']


In [2]:
import pandas as pd
import numpy as np

def clean_and_engineer(df, is_train=True):
    df = df.copy()
    
    if is_train:
        # 1. Data Hygiene: Remove zero or negative fares and distances if columns exist
        if 'total_amount' in df.columns:
            df = df[df['total_amount'] > 0].reset_index(drop=True)
        if 'trip_distance' in df.columns:
            df = df[df['trip_distance'] > 0].reset_index(drop=True)
            
            # Clip extreme fare/distance outliers in training data (99.9th percentile)
            fare_cap = df['total_amount'].quantile(0.999)
            df = df[df['total_amount'] <= fare_cap].reset_index(drop=True)

    # 2. Temporal Features (Extracting from pickup time)
    # Note: Change 'pickup_datetime' below to match your exact datetime column name if it's different
    datetime_col = 'pickup_datetime' if 'pickup_datetime' in df.columns else df.columns[1] # fallback
    
    if datetime_col in df.columns:
        df[datetime_col] = pd.to_datetime(df[datetime_col])
        df['hour'] = df[datetime_col].dt.hour
        df['dayofweek'] = df[datetime_col].dt.dayofweek
        df['month'] = df[datetime_col].dt.month
        df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
        
        # Rush hour indicator (e.g., Weekdays 8-10 AM and 5-8 PM)
        df['is_rush_hour'] = (
            (~df['is_weekend'].astype(bool)) & 
            ((df['hour'].between(8, 10)) | (df['hour'].between(17, 20)))
        ).astype(int)

    return df

# Apply to train and test
train_clean = clean_and_engineer(train_df, is_train=True)
test_clean = clean_and_engineer(test_df, is_train=False)

print(f"Cleaned Train Shape: {train_clean.shape}")
print(f"Cleaned Test Shape: {test_clean.shape}")

Cleaned Train Shape: (7798, 22)
Cleaned Test Shape: (2000, 22)


In [3]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

target = 'total_amount'

# 1. Find features that exist in BOTH train and test clean datasets (ignoring target and id)
exclude_cols = [target, 'id', 'pickup_datetime', 'dropoff_datetime']
train_numeric = train_clean.select_dtypes(include=[np.number]).columns.tolist()
test_numeric = test_clean.select_dtypes(include=[np.number]).columns.tolist()

# Intersect to ensure features exist in both sets
features = [col for col in train_numeric if col in test_numeric and col not in exclude_cols]

print(f"Training using {len(features)} shared numeric features: {features}")

# 2. Align train and test features precisely and cast them to float32 to prevent DTypePromotionError
X = train_clean[features].astype(np.float32)
y = train_clean[target].astype(np.float32)
X_test = test_clean[features].astype(np.float32)

# 3. Set up K-Fold Cross-Validation
SEED = 42
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

# 4. Training Loop
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        random_state=SEED,
        verbose=-1
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / kf.n_splits

# 5. Evaluate Overall Performance
cv_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\nOverall Out-of-Fold RMSE: {cv_rmse:.4f}")

# 6. Create Submission File
id_col = 'id' if 'id' in test_clean.columns else test_clean.columns[0]

submission = pd.DataFrame({
    id_col: test_clean[id_col],
    'total_amount': test_preds
})

submission.to_csv('submission.csv', index=False)
print("\nSubmission file 'submission.csv' generated successfully!")
display(submission.head())

Training using 16 shared numeric features: ['passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'extra', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'Airport_fee', 'hour', 'dayofweek', 'month', 'is_weekend', 'is_rush_hour']

Overall Out-of-Fold RMSE: 4.8350

Submission file 'submission.csv' generated successfully!


,id,total_amount
0,0,29.073200
1,1,8.823122
2,2,18.874594
3,3,19.847871
4,4,18.297987
